> **Superseded intermediate step.** This notebook produced the first labelled dataset (English only: 2,000 cleaned posts, 1,217 relevant after removing irrelevant ones). The posts were later merged with the Malay/Manglish and extra collections (`src/merge_v3.py`, notebook 04). The **final labelled file** is `data/grab_sentiment_clean.csv` (**2,015 posts**: negative 899 / 44.6%, neutral 898 / 44.6%, positive 218 / 10.8%), used by `06_model_comparison.ipynb` and `05_visualisation_and_findings.ipynb`.
>
> It still runs top to bottom for reference. Saving is switched off (`SAVE_OUTPUTS = False`) so the original files are not overwritten.

# 03 · Manual Labelling
**Input:** `data/processed/FINAL_grab_my_posts_clean_english.csv` (all 2,000 cleaned posts from notebook 02)
**Outputs:** `data/labelling/labels_for_review.csv` (Part 1) → `data/processed/labelled_dataset.csv` (Part 2)

Every post in the cleaned dataset gets a sentiment label. This notebook has two parts:

| Part | When | What happens |
|---|---|---|
| **1. Labels → review file** | now | **All 2,000 posts** were labelled by Claude, reading each post against `data/LABELLING_GUIDE.md` in batches of about 50 (**no VADER or keyword rules**). The labels are combined into a review file for a teammate to cross-check. |
| **2. Apply the review** | after the teammate returns `labels_reviewed.csv` | The teammate's corrections replace the original labels. "Irrelevant" posts are removed, and the labelled dataset is saved. |

**Labels:** `positive`, `negative`, `neutral`, `irrelevant` (not about Grab Malaysia). See the guide for the rules
(mixed posts = main feeling, sarcasm = intended meaning, news = neutral unless the poster adds an opinion, …).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import collect, label

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

cfg = collect.load_config()
processed = collect.project_path(cfg["paths"]["processed"])
tables = collect.project_path(cfg["paths"]["tables"])
labelling = collect.project_path("data/labelling")

main = pd.read_csv(processed / cfg["cleaning"]["output_file"])
main["thread_title"] = label.thread_titles(main)   # Reddit comments: the thread they reply to
print(f"Cleaned dataset: {len(main):,} posts")

Cleaned dataset: 2,000 posts


In [2]:
SAVE_OUTPUTS = False   # True would overwrite labels_for_review.csv, labels_reviewed.csv and labelled_dataset.csv

---
## Part 1 · Labels and the review file

### How the labels were made
- Every post was read in full and labelled against the guide, in batches of about 50. Progress was saved to
  `data/labelling/claude_labels.csv` after each batch, so the work could resume if interrupted.
- Reddit comments were read together with their **thread title**, because a reply like "same thing happened to me"
  only makes sense next to it. Batches kept each thread's comments together.
- Where two labels were reasonable, the post is marked **unsure** with a short reason.

The review file lists the **unsure posts first**, so the checker sees the hardest cases first. Each row has a
`url` link to the original post, which shows the images, replies and full thread.

In [3]:
claude_labels = pd.read_csv(labelling / "claude_labels.csv", keep_default_na=False)
# Safety check: exactly one label for every cleaned post
assert claude_labels["post_id"].is_unique, "Duplicate post_ids in claude_labels.csv"
missing = set(main["post_id"]) - set(claude_labels["post_id"])
assert not missing, f"{len(missing)} cleaned posts have no label yet"

review = label.build_review_file(main, claude_labels)
if SAVE_OUTPUTS:   # off: keep the saved files from the original run
    review.to_csv(labelling / "labels_for_review.csv", index=False)
    print("Saved data/labelling/labels_for_review.csv")
print(f"Unsure posts (listed first): {review['unsure'].sum()} of {len(review):,}")
display(label.label_distribution(review["label"]).rename_axis("Claude's labels"))
pd.crosstab(review["platform"], review["label"], margins=True, margins_name="TOTAL")

Unsure posts (listed first): 354 of 2,000


,count,percent
Claude's labels,,
positive,157,7.8
negative,452,22.6
neutral,613,30.6
irrelevant,778,38.9


label,irrelevant,negative,neutral,positive,TOTAL
platform,,,,,
reddit,499,236,371,63,1169
threads,2,27,3,7,39
twitter,277,189,239,87,792
TOTAL,778,452,613,157,2000


A few examples of each label:

In [4]:
for lab, grp in review.groupby("label"):
    print(f"\n===== {lab.upper()} =====")
    for _, r in grp.sample(min(4, len(grp)), random_state=1).iterrows():
        print(f"  [{r.platform}] {r.text[:170]}")


===== IRRELEVANT =====
  [reddit] KFC helpline here. Tq but we dont care about you.
  [reddit] In theory ok, in practice maybe. Fine. No. Think of Duitnow as local interbank/wallet transfer. Yes to card.
  [twitter] Another #Raiders training camp practice is in the books. Here are a few takeaways: - Caleb Rogers and Jackson Powers-Johnson alternated at RG with the 1s. JPJ also had so
  [twitter] The Bhumjaithai Party's flood-relief kitchen had earned a 4.8 out of 5 rating on food delivery platform GrabFood as of Friday. Most reviewers praised the hot food and gen

===== NEGATIVE =====
  [reddit] Hopefully still have proper CS team for escalations. Sounds like not tho.
  [twitter] Predator Grab earning blood money? @GrabMY
  [twitter] @user No need for Grab. Leave Sarawak alone
  [reddit] Actually searched up this post and this guy asked my gf the same thing too and he didnt say or show any gratitude we think this is a scam and this post kind of proves as 

===== NEUTRAL =====
  [reddi

### Instructions for the second checker
1. Open `data/labelling/labels_for_review.csv` in Excel or Google Sheets.
2. Read `LABELLING_GUIDE.md` first.
3. For each post (start at the top: unsure ones first), read `text` (and `thread_title` for Reddit comments).
   Click `url` to open the original post if you need more context (images, replies, the full thread).
4. If you **disagree** with `label`, type the correct label in **`corrected_label`**; leave it empty if you agree.
   Use exactly: `positive`, `negative`, `neutral`, `irrelevant`.
5. Save as **`data/labelling/labels_reviewed.csv`** (same columns), then run Part 2 below.

If there isn't time for all 2,000, check at least the unsure posts at the top of the file.

---
## Part 2 · Apply the review, remove irrelevant posts, save

If `labels_reviewed.csv` doesn't exist yet, Part 2 runs on Claude's labels alone and says so clearly.
Those results are **provisional**: re-run this notebook after the review comes back.

### Merge the checker's file
The checker may return a file that covers only part of the posts, e.g. an earlier 700-post version downloaded from
Google Sheets as `labels_for_review - labels_for_review.csv`. The cell below copies every `corrected_label` they
filled in into the full 2,000-post review file, matched on `post_id`, and fixes small spelling slips (e.g.
"irrlevant"). Posts they didn't check keep Claude's label. The result is saved as `labels_reviewed.csv`.

In [5]:
returned = sorted(labelling.glob("labels_for_review - *.csv"))   # files exported from Google Sheets
if returned:
    merged = review.copy()
    for f in returned:
        merged, notes = label.merge_review(merged, pd.read_csv(f, keep_default_na=False))
        print(f"Merged corrections from: {f.name}")
        for n in notes:
            print("  spelling fixed ->", n)
    if SAVE_OUTPUTS:   # off: keep the saved file
        merged.to_csv(labelling / "labels_reviewed.csv", index=False)
    print(f"Merged review: {len(merged):,} posts, {(merged['corrected_label'] != '').sum()} with a checker's label"
          + ("" if SAVE_OUTPUTS else " (not saved: SAVE_OUTPUTS = False)"))
else:
    print("No returned checker file found yet.")

Merged corrections from: labels_for_review - labels_for_review.csv
  spelling fixed -> oob7lm8: 'irrlevant' read as 'irrelevant'
Merged review: 2,000 posts, 23 with a checker's label (not saved: SAVE_OUTPUTS = False)


In [6]:
reviewed_path = labelling / "labels_reviewed.csv"
if reviewed_path.exists():
    reviewed = pd.read_csv(reviewed_path, keep_default_na=False)
    print("Using the reviewed file: labels_reviewed.csv")
else:
    reviewed = review.copy()
    print("labels_reviewed.csv not found: results below are PROVISIONAL (Claude's labels only).")

final, n_changed = label.apply_corrections(reviewed)
print(f"Labels changed by the reviewer: {n_changed} of {len(final):,} ({n_changed/len(final):.1%})")
checked = (final["corrected_label"].astype(str).str.strip() != "").sum()
print(f"Posts the checker filled in: {checked} (the rest keep Claude's label)")
if n_changed:
    display(pd.crosstab(final["label"], final["final_label"], rownames=["Claude"], colnames=["after review"]))
    display(final.loc[final["final_label"] != final["label"], ["platform", "label", "final_label", "text"]]
            .rename(columns={"label": "Claude", "final_label": "checker"})
            .assign(text=lambda d: d["text"].str.slice(0, 120)))

Using the reviewed file: labels_reviewed.csv
Labels changed by the reviewer: 12 of 2,000 (0.6%)
Posts the checker filled in: 23 (the rest keep Claude's label)


after review,irrelevant,negative,neutral,positive
Claude,,,,
irrelevant,777,0,1,0
negative,1,451,0,0
neutral,4,3,605,1
positive,1,0,1,155


,platform,Claude,checker,text
11,reddit,neutral,negative,"just another one sided video, while the one recording is not in the video. that's convenience. I really hate this kind o"
25,reddit,positive,irrelevant,That's like asking what are the benefits shoppee/lazada give to the sellers. Linking drivers to passengers. Are you too
41,reddit,neutral,irrelevant,Open letter to Temasek Holdings from Malaysian Grab driver
63,reddit,negative,irrelevant,Why more merchants rejecting Grabpay and only accepts TNG payments? Anyone doing business care to explain why? Does grab
70,reddit,neutral,negative,Don't waste time taking Grab from the airport. You will end up waiting in the queue and also will be stuck in the mornin
71,reddit,neutral,positive,"Some game channel also dont accept grabpay. Old time can, now cannot. Dont know why. I like the grab coins benefit too."
108,reddit,neutral,irrelevant,On the way
118,reddit,neutral,irrelevant,"URL Found this in the thread though. But the best case scenario, let them check the CCTV footage on it."
121,reddit,neutral,irrelevant,Family mart is notorious for understaffing and food delivery orders takes up to an hour or more to be packed up.This is
146,reddit,neutral,negative,Yes. Legally speaking this man is right. Yes very anti consumer. But base on legal this is the right answer


### Remove "irrelevant" posts
Every post is labelled, so this removes all irrelevant posts from the dataset. How many were removed is logged.

In [7]:
labelled = final[final["final_label"] != "irrelevant"]
removal_log = pd.DataFrame({
    "platform": ["reddit", "twitter", "threads", "TOTAL"],
    "before": [int((final["platform"] == p).sum()) for p in ["reddit", "twitter", "threads"]] + [len(final)],
    "after": [int((labelled["platform"] == p).sum()) for p in ["reddit", "twitter", "threads"]] + [len(labelled)],
})
removal_log["irrelevant_removed"] = removal_log["before"] - removal_log["after"]
if SAVE_OUTPUTS:   # off: keep the saved files from the original run
    removal_log.to_csv(tables / "irrelevant_removed_log.csv", index=False)
print(f"Removed {len(final) - len(labelled):,} irrelevant posts; {len(labelled):,} relevant posts remain.")
removal_log

Removed 783 irrelevant posts; 1,217 relevant posts remain.


,platform,before,after,irrelevant_removed
0,reddit,1169,665,504
1,twitter,792,515,277
2,threads,39,37,2
3,TOTAL,2000,1217,783


In [8]:
# Labelled dataset: the reviewed label plus all columns from the cleaned dataset (text_clean, text_norm, …)
labelled_out = (labelled[["post_id", "final_label", "label", "corrected_label", "unsure", "reason"]]
                .rename(columns={"final_label": "sentiment", "label": "claude_label"})
                .merge(main.drop(columns="thread_title"), on="post_id", how="left", validate="1:1"))
if SAVE_OUTPUTS:   # off: keep the saved files from the original run
    labelled_out.to_csv(processed / "labelled_dataset.csv", index=False)
    print("Saved data/processed/labelled_dataset.csv", f"({len(labelled_out):,} posts)")
print(f"Labelled dataset (this run): {len(labelled_out):,} posts")

Labelled dataset (this run): 1,217 posts


### Final label distribution

In [9]:
dist = label.label_distribution(labelled_out["sentiment"])
if SAVE_OUTPUTS:   # off: keep the saved files from the original run
    dist.to_csv(tables / "label_distribution.csv")
    print("Saved outputs/tables/label_distribution.csv")
display(dist)
pd.crosstab(labelled_out["platform"], labelled_out["sentiment"], margins=True, margins_name="TOTAL")

,count,percent
sentiment,,
positive,156,12.8
negative,454,37.3
neutral,607,49.9


sentiment,negative,neutral,positive,TOTAL
platform,,,,
reddit,238,365,62,665
threads,27,3,7,37
twitter,189,239,87,515
TOTAL,454,607,156,1217


---
**Next step:** `labelled_dataset.csv` was merged with the later Malay/Manglish and extra collections into `labelled_dataset_v3.csv` (`04_sentiment_analysis.ipynb`, `src/merge_v3.py`). The final file `data/grab_sentiment_clean.csv` (2,015 posts) is analysed in `06_model_comparison.ipynb` (VADER vs Logistic Regression) and `05_visualisation_and_findings.ipynb` (charts).